# Starlar Dataset Inspection for LLM Fine-Tuning v2

This notebook inspects the CENG493_Starlar dataset to decide whether it is suitable for LLM supervised fine-tuning.

Goal:
- Extract the Starlar dataset.
- Inspect file structure and columns.
- Determine whether it contains question + answer + context fields.
- If suitable, prepare a cleaner SFT dataset for a second QLoRA fine-tuning run.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import glob
import zipfile
import pandas as pd
import json
import numpy as np

project_path = "/content/drive/MyDrive/turkish_legal_rag"

raw_external_path = f"{project_path}/data/raw/external_datasets"
starlar_zip_path = f"{raw_external_path}/CENG493_Starlar.zip"

starlar_extract_path = f"{raw_external_path}/extracted/CENG493_Starlar"

processed_path = f"{project_path}/data/processed"
outputs_path = f"{project_path}/outputs"
metrics_path = f"{outputs_path}/metrics"

print("Starlar zip exists:", os.path.exists(starlar_zip_path))
print("Zip path:", starlar_zip_path)
print("Extract path:", starlar_extract_path)

Starlar zip exists: True
Zip path: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/CENG493_Starlar.zip
Extract path: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar


In [3]:
os.makedirs(starlar_extract_path, exist_ok=True)

with zipfile.ZipFile(starlar_zip_path, "r") as zip_ref:
    zip_ref.extractall(starlar_extract_path)

print("Extracted to:", starlar_extract_path)

Extracted to: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar


In [4]:
for root, dirs, files in os.walk(starlar_extract_path):
    level = root.replace(starlar_extract_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:20]:
        file_path = os.path.join(root, file)
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"{indent}  - {file} | {size_mb:.2f} MB")

CENG493_Starlar/
  - corpus.jsonl | 16.63 MB
  - embedding.jsonl | 4.48 MB
  - gold_benchmark.json | 0.67 MB
  - llm.jsonl | 33.56 MB
  - rag_eval.json | 1.80 MB
  - reranker.jsonl | 7.78 MB


In [5]:
starlar_root = starlar_extract_path

corpus_path = f"{starlar_root}/corpus.jsonl"
embedding_path = f"{starlar_root}/embedding.jsonl"
gold_benchmark_path = f"{starlar_root}/gold_benchmark.json"
llm_path = f"{starlar_root}/llm.jsonl"
rag_eval_path = f"{starlar_root}/rag_eval.json"
reranker_path = f"{starlar_root}/reranker.jsonl"

paths = {
    "corpus": corpus_path,
    "embedding": embedding_path,
    "gold_benchmark": gold_benchmark_path,
    "llm": llm_path,
    "rag_eval": rag_eval_path,
    "reranker": reranker_path
}

for name, path in paths.items():
    print(name, os.path.exists(path), path)

corpus True /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/corpus.jsonl
embedding True /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/embedding.jsonl
gold_benchmark True /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/gold_benchmark.json
llm True /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/llm.jsonl
rag_eval True /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/rag_eval.json
reranker True /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/reranker.jsonl


In [6]:
import pandas as pd
import json
import os

def load_json_or_jsonl(path):
    if path.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    elif path.endswith(".json"):
        return pd.read_json(path)
    else:
        raise ValueError("Unsupported file type: " + path)


loaded_tables = {}

for name, path in paths.items():
    print("=" * 120)
    print("NAME:", name)
    print("PATH:", path)

    try:
        df = load_json_or_jsonl(path)
        loaded_tables[name] = df

        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(3))

    except Exception as e:
        print("Could not read:", repr(e))

NAME: corpus
PATH: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/corpus.jsonl
Shape: (7579, 4)
Columns: ['id', 'text', 'title', 'metadata']


,id,text,title,metadata
0,oricon_anayasa_000001,"Susma hakkı, kişinin kendi lehine veya aleyhin...",Anayasa Hukuku ve Temel Haklar,"{'source': 'ORICON', 'source_file': 'ORICON_cl..."
1,oricon_anayasa_000003,"Düşünce özgürlüğü, düşünce ve kanaatlerin çeşi...",Anayasa Hukuku ve Temel Haklar,"{'source': 'ORICON', 'source_file': 'ORICON_cl..."
2,oricon_anayasa_000004,"İfade özgürlüğü, demokratik toplumların temeli...",Anayasa Hukuku ve Temel Haklar,"{'source': 'ORICON', 'source_file': 'ORICON_cl..."


NAME: embedding
PATH: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/embedding.jsonl
Shape: (2059, 11)
Columns: ['id', 'query', 'positive_passage', 'negative_passage', 'positive_id', 'negative_id', 'positive_citation', 'negative_citation', 'negative_type', 'source', 'metadata']


,id,query,positive_passage,negative_passage,positive_id,negative_id,positive_citation,negative_citation,negative_type,source,metadata
0,emb_clean_000001,"Hukuk Genel Kurulu 2013/2239 E., 2015/1334 K. ...",Direnme yoluyla Hukuk Genel Kurulu önüne gelen...,II. CEVAP 1. Davalı... vekili cevap dilekçesin...,yargitay_1310_kamulastirma_hukuku_004,yargitay_0942_kamulastirma_hukuku_003,YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY - BELGE 1...,YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY - BELGE 0...,hard_negative_same_source_or_category,YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY,{'positive_source_file': 'YARGITAY_RETRIEVAL_c...
1,emb_clean_000005,"Hukuk Genel Kurulu 2024/235 E., 2025/211 K. sa...",Aynı ilke Yargıtay Hukuk Genel Kurulunun 24.09...,Hukuk Dairesi kararının da yok hükmünde olduğu...,yargitay_1107_medeni_usul_hukuku_005,yargitay_1020_medeni_usul_hukuku_005,YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY - BELGE 1...,YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY - BELGE 1...,hard_negative_same_source_or_category,YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY,{'positive_source_file': 'YARGITAY_RETRIEVAL_c...
2,emb_clean_000006,KAYIT 0697 kaydında Eğitim hakkı bakımından ol...,KAYIT 0697 | Karar sonucu: İhlal | Temel hak k...,KAYIT 0751 | Karar sonucu: İhlal | Temel hak k...,train_kayit_0697_001,train_kayit_0751_001,TRAIN_LOW_RISK_ONLY - KAYIT 0697 - İhlal - Eği...,TRAIN_LOW_RISK_ONLY - KAYIT 0751 - İhlal - Eği...,hard_negative_same_source_or_category,TRAIN_LOW_RISK_ONLY,{'positive_source_file': 'TRAIN_clean_source_f...


NAME: gold_benchmark
PATH: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/gold_benchmark.json
Shape: (240, 11)
Columns: ['question_id', 'question', 'verified_answer', 'gold_sources', 'answer_type', 'difficulty', 'benchmark_status', 'manual_legal_review_recommended', 'validation_flags', 'quality_profile', 'notes']


...

,id,messages,metadata
0,sft_exp_000001,"[{'role': 'system', 'content': 'Sen bir Türk h...","{'source_id': 'oricon_anayasa_000001', 'corpus..."
1,sft_exp_000002,"[{'role': 'system', 'content': 'Sen bir Türk h...","{'source_id': 'oricon_anayasa_000001', 'corpus..."
2,sft_exp_000003,"[{'role': 'system', 'content': 'Sen bir Türk h...","{'source_id': 'oricon_anayasa_000001', 'corpus..."


NAME: rag_eval
PATH: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/rag_eval.json
Shape: (1000, 11)
Columns: ['query_id', 'query', 'gold_chunk_ids', 'gold_citation_labels', 'source', 'source_file', 'title', 'category', 'semantic_topic', 'gold_answer_extract', 'metadata']


,query_id,query,gold_chunk_ids,gold_citation_labels,source,source_file,title,category,semantic_topic,gold_answer_extract,metadata
0,eval_clean_0001_oricon_genel_001293,Genel hukuk / sınıflandırma bekliyor alanında ...,[oricon_genel_001293],[ORICON - Genel hukuk / sınıflandırma bekliyor...,ORICON,ORICON_clean_legal_source_for_chunking.txt,Genel hukuk / sınıflandırma bekliyor,Genel hukuk / sınıflandırma bekliyor,Türkiye den günlük erişimi milyondan,Kaynağa göre: Türkiye'den günlük erişimi bir m...,"{'row_id': 'oricon_genel_001293', 'verificatio..."
1,eval_clean_0002_oricon_medeni_000093,Medeni Hukuk - Miras/Aile/Eşya/Kişiler alanınd...,[oricon_medeni_000093],[ORICON - Medeni Hukuk - Miras/Aile/Eşya/Kişil...,ORICON,ORICON_clean_legal_source_for_chunking.txt,Medeni Hukuk - Miras/Aile/Eşya/Kişiler,Medeni Hukuk - Miras/Aile/Eşya/Kişiler,Resmi vasiyetname düzenleyen kişi,Kaynağa göre: Resmi vasiyetname düzenleyen kiş...,"{'row_id': 'oricon_medeni_000093', 'verificati..."
2,eval_clean_0003_oricon_ticaret_000252,Ticaret Hukuku alanında Yurt dışında bulunanla...,[oricon_ticaret_000252],[ORICON - Ticaret Hukuku - oricon_ticaret_000252],ORICON,ORICON_clean_legal_source_for_chunking.txt,Ticaret Hukuku,Ticaret Hukuku,Yurt dışında bulunanlara tebliğ işlemi,Kaynağa göre: Yurt dışında bulunanlara tebliğ ...,"{'row_id': 'oricon_ticaret_000252', 'verificat..."


NAME: reranker
PATH: /content/drive/MyDrive/turkish_legal_rag/data/raw/external_datasets/extracted/CENG493_Starlar/reranker.jsonl
Shape: (6752, 10)
Columns: ['id', 'query_id', 'query', 'candidate_passage', 'label', 'candidate_id', 'citation_label', 'source', 'negative_type', 'audit_status']


,id,query_id,query,candidate_passage,label,candidate_id,citation_label,source,negative_type,audit_status
0,rerank_noleak_balanced_000001,rerank_query_final_000001,18 yaşından küçük çocuk hakkında ORICON kaynağ...,18 yaşından küçük bir çocuk için DNA testi yap...,1,oricon_genel_001370,ORICON - Genel hukuk / sınıflandırma bekliyor ...,ORICON,None,no_eval_gold_leak_balanced_v3
1,rerank_noleak_balanced_000002,rerank_query_final_000001,18 yaşından küçük çocuk hakkında ORICON kaynağ...,İdarenin takdir yetkisi kamu yararı ve hizmet ...,0,oricon_genel_000975,ORICON - Genel hukuk / sınıflandırma bekliyor ...,ORICON,hard_negative_same_source_or_category,no_eval_gold_leak_balanced_v3
2,rerank_noleak_balanced_000003,rerank_query_final_000001,18 yaşından küçük çocuk hakkında ORICON kaynağ...,Satış bedelinin tamamının peşin ödenmesi hâlin...,0,oricon_genel_001651,ORICON - Genel hukuk / sınıflandırma bekliyor ...,ORICON,hard_negative_same_source_or_category,no_eval_gold_leak_balanced_v3


In [7]:
llm_df = loaded_tables["llm"].copy()

print("LLM shape:", llm_df.shape)
print("LLM columns:", llm_df.columns.tolist())

display(llm_df.head(10))

print("\nSample record:")
print(llm_df.iloc[0].to_dict())

LLM shape: (13758, 3)
LLM columns: ['id', 'messages', 'metadata']


,id,messages,metadata
0,sft_exp_000001,"[{'role': 'system', 'content': 'Sen bir Türk h...","{'source_id': 'oricon_anayasa_000001', 'corpus..."
1,sft_exp_000002,"[{'role': 'system', 'content': 'Sen bir Türk h...","{'source_id': 'oricon_anayasa_000001', 'corpus..."
2,sft_exp_000003,"[{'role': 'system', 'content': 'Sen bir Türk h...","{'source_id': 'oricon_anayasa_000001', 'corpus..."
3,sft_exp_000004,"[{'role': 'system', 'content': 'Sen bir Türk h...","{'source_id': 'oricon_anayasa_000003', 'corpus..."
4,sft_exp_000005,"[{'role': 'system', 'content': 'Sen bir Türk h...","{'source_id': 'oricon_anayasa_000003', 'corpus..."
5,sft_exp_000006,"[{'role': 'system', 'content': 'Sen bir Türk h...","{'source_id': 'oricon_anayasa_000003', 'corpus..."
6,sft_exp_000007,"[{'role': 'system', 'content': 'Sen bir Türk h...","{'source_id': 'oricon_anayasa_000004', 'corpus..."
7,sft_exp_000009,"[{'role': 'system', 'content': 'Sen bir Türk h...","{'source_id': 'oricon_anayasa_000004', 'corpus..."
8,sft_exp_000016,"[{'role': 'system', 'content': 'Sen bir Türk h...","{'source_id': 'oricon_anayasa_000007', 'corpus..."
9,sft_exp_000017,"[{'role': 'system', 'content': 'Sen bir Türk h...","{'source_id': 'oricon_anayasa_000007', 'corpus..."



Sample record:
{'id': 'sft_exp_000001', 'messages': [{'role': 'system', 'content': 'Sen bir Türk hukuku RAG asistanısın. Yalnızca kullanıcı tarafından verilen kaynak metne dayanarak cevap ver. Kaynakta olmayan bilgiyi üretme. Cevabın sonunda kaynak/citation bilgisini mutlaka belirt. Kaynak kesin resmî karar metni değilse bunu kesin hüküm gibi sunma.'}, {'role': 'user', 'content': '[Kaynak]\nBaşlık: Anayasa Hukuku ve Temel Haklar\nKaynak: ORICON\nDosya: ORICON_clean_legal_source_for_chunking.txt\nChunk ID: oricon_anayasa_000001\nCitation: ORICON - Anayasa Hukuku ve Temel Haklar - oricon_anayasa_000001\nMetin: Susma hakkı, kişinin kendi lehine veya aleyhine ifade verme kararını özgürce kullanabilmesine olanak tanır.\n\nSoru: susma hakkı hakkında verilen kaynağa göre ne söylenebilir?'}, {'role': 'assistant', 'content': 'Kaynağa göre: Susma hakkı, kişinin kendi lehine veya aleyhine ifade verme kararını özgürce kullanabilmesine olanak tanır.\n\nKaynak: ORICON - Anayasa Hukuku ve Temel Hakl

In [8]:
def get_roles(messages):
    try:
        return [m.get("role") for m in messages]
    except:
        return []


llm_df["roles"] = llm_df["messages"].apply(get_roles)
llm_df["roles_str"] = llm_df["roles"].apply(lambda x: " > ".join(x))

role_counts = llm_df["roles_str"].value_counts().reset_index()
role_counts.columns = ["roles", "count"]

display(role_counts)

,roles,count
0,system > user > assistant,13758


In [9]:
def extract_message_content(messages, role):
    try:
        for m in messages:
            if m.get("role") == role:
                return str(m.get("content", "")).strip()
    except:
        return ""
    return ""


llm_df["system_content"] = llm_df["messages"].apply(lambda x: extract_message_content(x, "system"))
llm_df["user_content"] = llm_df["messages"].apply(lambda x: extract_message_content(x, "user"))
llm_df["assistant_content"] = llm_df["messages"].apply(lambda x: extract_message_content(x, "assistant"))

print("Empty system:", (llm_df["system_content"].str.len() == 0).sum())
print("Empty user:", (llm_df["user_content"].str.len() == 0).sum())
print("Empty assistant:", (llm_df["assistant_content"].str.len() == 0).sum())

display(llm_df[["system_content", "user_content", "assistant_content"]].head(3))

Empty system: 0
Empty user: 0
Empty assistant: 0


,system_content,user_content,assistant_content
0,Sen bir Türk hukuku RAG asistanısın. Yalnızca ...,[Kaynak]\nBaşlık: Anayasa Hukuku ve Temel Hakl...,"Kaynağa göre: Susma hakkı, kişinin kendi lehin..."
1,Sen bir Türk hukuku RAG asistanısın. Yalnızca ...,[Kaynak]\nBaşlık: Anayasa Hukuku ve Temel Hakl...,"Özet: Susma hakkı, kişinin kendi lehine veya a..."
2,Sen bir Türk hukuku RAG asistanısın. Yalnızca ...,[Kaynak]\nBaşlık: Anayasa Hukuku ve Temel Hakl...,Bu açıklama şu kaynak bilgilerine dayandırılma...


In [10]:
llm_df["system_len"] = llm_df["system_content"].str.len()
llm_df["user_len"] = llm_df["user_content"].str.len()
llm_df["assistant_len"] = llm_df["assistant_content"].str.len()

display(llm_df[["system_len", "user_len", "assistant_len"]].describe())

print("Very long user prompts > 3500 chars:", (llm_df["user_len"] > 3500).sum())
print("Very long assistant answers > 1200 chars:", (llm_df["assistant_len"] > 1200).sum())

,system_len,user_len,assistant_len
count,13758.0,13758.000000,13758.000000
mean,267.0,834.390391,583.009885
std,0.0,602.778543,508.798985
min,267.0,331.000000,143.000000
25%,267.0,470.000000,271.000000
50%,267.0,583.000000,385.000000
75%,267.0,882.000000,647.750000
max,267.0,3621.000000,2950.000000


Very long user prompts > 3500 chars: 6
Very long assistant answers > 1200 chars: 1650


In [11]:
def metadata_get(metadata, key):
    try:
        if isinstance(metadata, dict):
            return metadata.get(key)
    except:
        pass
    return None


llm_df["metadata_source"] = llm_df["metadata"].apply(lambda x: metadata_get(x, "source"))
llm_df["metadata_source_id"] = llm_df["metadata"].apply(lambda x: metadata_get(x, "source_id"))
llm_df["metadata_category"] = llm_df["metadata"].apply(lambda x: metadata_get(x, "category"))
llm_df["metadata_variant_type"] = llm_df["metadata"].apply(lambda x: metadata_get(x, "variant_type"))

print("Source counts:")
display(llm_df["metadata_source"].value_counts(dropna=False).head(20))

print("Variant type counts:")
display(llm_df["metadata_variant_type"].value_counts(dropna=False).head(20))

Source counts:


,count
metadata_source,
ORICON,8694
TURKISH_LAW_ESKI_LOW_RISK_ONLY,2912
YARGITAY_HGK_RETRIEVAL_LOW_RISK_ONLY,1316
TRAIN_LOW_RISK_ONLY,479
TURKISH_LAWCHATBOT_LOW_RISK_ONLY,357


Variant type counts:


,count
metadata_variant_type,
direct_grounded_answer,6339
source_limited_explanation,4436
short_summary_with_citation,2983


In [12]:
clean_llm_df = llm_df.copy()

clean_llm_df = clean_llm_df[
    (clean_llm_df["system_content"].str.len() > 5) &
    (clean_llm_df["user_content"].str.len() > 20) &
    (clean_llm_df["assistant_content"].str.len() > 5)
].copy()

# Çok uzun inputları atıyoruz ki model yine hukuk metni devam ettirmeye kaymasın
clean_llm_df = clean_llm_df[
    (clean_llm_df["user_len"] <= 3500) &
    (clean_llm_df["assistant_len"] <= 1200)
].copy()

print("Original rows:", len(llm_df))
print("Clean rows:", len(clean_llm_df))
print("Removed rows:", len(llm_df) - len(clean_llm_df))

display(clean_llm_df[[
    "system_content",
    "user_content",
    "assistant_content",
    "user_len",
    "assistant_len"
]].head(3))

Original rows: 13758
Clean rows: 12108
Removed rows: 1650


,system_content,user_content,assistant_content,user_len,assistant_len
0,Sen bir Türk hukuku RAG asistanısın. Yalnızca ...,[Kaynak]\nBaşlık: Anayasa Hukuku ve Temel Hakl...,"Kaynağa göre: Susma hakkı, kişinin kendi lehin...",399,194
1,Sen bir Türk hukuku RAG asistanısın. Yalnızca ...,[Kaynak]\nBaşlık: Anayasa Hukuku ve Temel Hakl...,"Özet: Susma hakkı, kişinin kendi lehine veya a...",393,186
2,Sen bir Türk hukuku RAG asistanısın. Yalnızca ...,[Kaynak]\nBaşlık: Anayasa Hukuku ve Temel Hakl...,Bu açıklama şu kaynak bilgilerine dayandırılma...,417,311


In [13]:
def build_mistral_sft_text_from_messages(system_content, user_content, assistant_content):
    system_content = str(system_content).strip()
    user_content = str(user_content).strip()
    assistant_content = str(assistant_content).strip()

    return f"""<s>[INST] {system_content}

{user_content} [/INST] {assistant_content}</s>"""


clean_llm_df["text"] = clean_llm_df.apply(
    lambda row: build_mistral_sft_text_from_messages(
        row["system_content"],
        row["user_content"],
        row["assistant_content"]
    ),
    axis=1
)

print(clean_llm_df.iloc[0]["text"][:2500])

<s>[INST] Sen bir Türk hukuku RAG asistanısın. Yalnızca kullanıcı tarafından verilen kaynak metne dayanarak cevap ver. Kaynakta olmayan bilgiyi üretme. Cevabın sonunda kaynak/citation bilgisini mutlaka belirt. Kaynak kesin resmî karar metni değilse bunu kesin hüküm gibi sunma.

[Kaynak]
Başlık: Anayasa Hukuku ve Temel Haklar
Kaynak: ORICON
Dosya: ORICON_clean_legal_source_for_chunking.txt
Chunk ID: oricon_anayasa_000001
Citation: ORICON - Anayasa Hukuku ve Temel Haklar - oricon_anayasa_000001
Metin: Susma hakkı, kişinin kendi lehine veya aleyhine ifade verme kararını özgürce kullanabilmesine olanak tanır.

Soru: susma hakkı hakkında verilen kaynağa göre ne söylenebilir? [/INST] Kaynağa göre: Susma hakkı, kişinin kendi lehine veya aleyhine ifade verme kararını özgürce kullanabilmesine olanak tanır.

Kaynak: ORICON - Anayasa Hukuku ve Temel Haklar - oricon_anayasa_000001</s>


In [14]:
clean_llm_df["text_len"] = clean_llm_df["text"].str.len()

display(clean_llm_df[["text_len", "user_len", "assistant_len"]].describe())

print("Texts > 5000 chars:", (clean_llm_df["text_len"] > 5000).sum())

,text_len,user_len,assistant_len
count,12108.000000,12108.000000,12108.000000
mean,1351.688553,643.258176,416.430377
std,470.956067,270.046044,209.154534
min,767.000000,331.000000,143.000000
25%,1025.000000,460.000000,258.000000
50%,1190.000000,548.000000,357.000000
75%,1526.000000,729.000000,510.000000
max,3410.000000,1948.000000,1200.000000


Texts > 5000 chars: 0


In [15]:
MAX_TOTAL_ROWS = min(len(clean_llm_df), 4000)

starlar_llm_v2_df = clean_llm_df.sample(
    n=MAX_TOTAL_ROWS,
    random_state=42
).reset_index(drop=True)

print("Selected rows for LLM v2:", len(starlar_llm_v2_df))

display(starlar_llm_v2_df[[
    "id",
    "text",
    "metadata_source",
    "metadata_variant_type"
]].head(3))

Selected rows for LLM v2: 4000


,id,text,metadata_source,metadata_variant_type
0,sft_exp_002113,<s>[INST] Sen bir Türk hukuku RAG asistanısın....,ORICON,direct_grounded_answer
1,sft_exp_011856,<s>[INST] Sen bir Türk hukuku RAG asistanısın....,TURKISH_LAW_ESKI_LOW_RISK_ONLY,source_limited_explanation
2,sft_exp_010998,<s>[INST] Sen bir Türk hukuku RAG asistanısın....,ORICON,source_limited_explanation


In [16]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    starlar_llm_v2_df,
    test_size=0.2,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (3200, 17)
Val: (400, 17)
Test: (400, 17)


In [17]:
starlar_llm_v2_train_path = f"{processed_path}/starlar_llm_sft_v2_train.jsonl"
starlar_llm_v2_val_path = f"{processed_path}/starlar_llm_sft_v2_val.jsonl"
starlar_llm_v2_test_path = f"{processed_path}/starlar_llm_sft_v2_test.jsonl"

columns_to_save = [
    "id",
    "text",
    "system_content",
    "user_content",
    "assistant_content",
    "metadata_source",
    "metadata_variant_type",
    "metadata_source_id"
]

train_df[columns_to_save].to_json(
    starlar_llm_v2_train_path,
    orient="records",
    lines=True,
    force_ascii=False
)

val_df[columns_to_save].to_json(
    starlar_llm_v2_val_path,
    orient="records",
    lines=True,
    force_ascii=False
)

test_df[columns_to_save].to_json(
    starlar_llm_v2_test_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved:")
print(starlar_llm_v2_train_path)
print(starlar_llm_v2_val_path)
print(starlar_llm_v2_test_path)

Saved:
/content/drive/MyDrive/turkish_legal_rag/data/processed/starlar_llm_sft_v2_train.jsonl
/content/drive/MyDrive/turkish_legal_rag/data/processed/starlar_llm_sft_v2_val.jsonl
/content/drive/MyDrive/turkish_legal_rag/data/processed/starlar_llm_sft_v2_test.jsonl


In [18]:
summary_df = pd.DataFrame([{
    "source_file": llm_path,
    "original_rows": len(llm_df),
    "clean_rows": len(clean_llm_df),
    "selected_rows": len(starlar_llm_v2_df),
    "train_rows": len(train_df),
    "val_rows": len(val_df),
    "test_rows": len(test_df),
    "max_user_len": 3500,
    "max_assistant_len": 1200,
    "train_path": starlar_llm_v2_train_path,
    "val_path": starlar_llm_v2_val_path,
    "test_path": starlar_llm_v2_test_path
}])

summary_path = f"{metrics_path}/starlar_llm_sft_v2_data_preparation_summary.csv"

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

display(summary_df)

print("Summary saved:", summary_path)

,source_file,original_rows,clean_rows,selected_rows,train_rows,val_rows,test_rows,max_user_len,max_assistant_len,train_path,val_path,test_path
0,/content/drive/MyDrive/turkish_legal_rag/data/...,13758,12108,4000,3200,400,400,3500,1200,/content/drive/MyDrive/turkish_legal_rag/data/...,/content/drive/MyDrive/turkish_legal_rag/data/...,/content/drive/MyDrive/turkish_legal_rag/data/...


Summary saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/starlar_llm_sft_v2_data_preparation_summary.csv


In [19]:
for path in [
    starlar_llm_v2_train_path,
    starlar_llm_v2_val_path,
    starlar_llm_v2_test_path,
    summary_path
]:
    print(path)
    print("Exists:", os.path.exists(path))
    if os.path.exists(path):
        print("Size MB:", round(os.path.getsize(path) / (1024 * 1024), 2))
    print("-" * 80)

/content/drive/MyDrive/turkish_legal_rag/data/processed/starlar_llm_sft_v2_train.jsonl
Exists: True
Size MB: 9.43
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/data/processed/starlar_llm_sft_v2_val.jsonl
Exists: True
Size MB: 1.23
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/data/processed/starlar_llm_sft_v2_test.jsonl
Exists: True
Size MB: 1.19
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/starlar_llm_sft_v2_data_preparation_summary.csv
Exists: True
Size MB: 0.0
--------------------------------------------------------------------------------
